In [1]:
import glob
import os
import pandas as pd

files = sorted(glob.glob("../data_raw/SDR-*.csv"))

schemas = {}
for f in files:
    name = os.path.basename(f)
    cols = pd.read_csv(f, nrows=0).columns.tolist()
    schemas[name] = cols
    print(f"{name}: {len(cols)} عمود")

first_file = list(schemas.keys())[0]
first_cols = schemas[first_file]

all_match = True
for name, cols in schemas.items():
    if cols != first_cols:
        all_match = False
        diff_added = set(cols) - set(first_cols)
        diff_missing = set(first_cols) - set(cols)
        print(f"\n⚠️ اختلاف في ملف {name}:")
        if diff_added: print(f"   أعمدة زائدة: {diff_added}")
        if diff_missing: print(f"   أعمدة ناقصة: {diff_missing}")

if all_match:
    print("\n✅ ممتاز! جميع الملفات متطابقة 100% في أسماء وترتيب الأعمدة وجاهزة للدمج.")
    print("\nأسماء الأعمدة:")
    for i, c in enumerate(first_cols, 1):
        print(f"{i}. {c}")

SDR-2019.csv: 76 عمود
SDR-2020.csv: 76 عمود
SDR-2021.csv: 76 عمود
SDR-2022.csv: 76 عمود
SDR-2023.csv: 76 عمود
SDR-2024.csv: 76 عمود
SDR-2025.csv: 76 عمود
SDR-2026.csv: 76 عمود

✅ ممتاز! جميع الملفات متطابقة 100% في أسماء وترتيب الأعمدة وجاهزة للدمج.

أسماء الأعمدة:
1. OperatorControlNumber
2. DifficultyDate
3. SubmissionDate
4. OperatorDesignator
5. SubmitterDesignator
6. SubmitterTypeCode
7. ReceivingRegionCode
8. ReceivingDistrictOffice
9. SDRType
10. JASCCode
11. NatureOfConditionA
12. NatureOfConditionB
13. NatureOfConditionC
14. PrecautionaryProcedureA
15. PrecautionaryProcedureB
16. PrecautionaryProcedureC
17. PrecautionaryProcedureD
18. StageOfOperationCode
19. HowDiscoveredCode
20. RegistryNNumber
21. AircraftMake
22. AircraftModel
23. AircraftSerialNumber
24. AircraftTotalTime
25. AircraftTotalCycles
26. EngineMake
27. EngineModel
28. EngineSerialNumber
29. EngineTotalTime
30. EngineTotalCycles
31. PropellerMake
32. PropellerModel
33. PropellerSerialNumber
34. PropellerTotalTi

In [2]:
# دمج الـ 8 ملفات
df = pd.concat([pd.read_csv(f, low_memory=False) for f in files], ignore_index=True)
print(f"إجمالي عدد الصفوف: {len(df):,}")

# طباعة أسماء كل الـ 76 عمود ونسبة القيم الفاضية في كل عمود
pd.set_option('display.max_rows', 100)
summary = pd.DataFrame({
    'Null_%': (df.isnull().sum() / len(df) * 100).round(1),
    'Sample': df.apply(lambda x: x.dropna().iloc[0] if not x.dropna().empty else 'EMPTY')
})
print(summary)

إجمالي عدد الصفوف: 496,203
                         Null_%  \
OperatorControlNumber       0.0   
DifficultyDate              0.0   
SubmissionDate              0.0   
OperatorDesignator          2.2   
SubmitterDesignator         1.8   
SubmitterTypeCode           0.0   
ReceivingRegionCode         0.0   
ReceivingDistrictOffice     0.0   
SDRType                     0.0   
JASCCode                    0.0   
NatureOfConditionA          0.0   
NatureOfConditionB         98.0   
NatureOfConditionC         99.7   
PrecautionaryProcedureA     0.0   
PrecautionaryProcedureB    99.1   
PrecautionaryProcedureC    99.8   
PrecautionaryProcedureD    99.9   
StageOfOperationCode        0.0   
HowDiscoveredCode           0.0   
RegistryNNumber             0.8   
AircraftMake                0.1   
AircraftModel               0.2   
AircraftSerialNumber        0.7   
AircraftTotalTime           1.2   
AircraftTotalCycles         2.2   
EngineMake                 96.6   
EngineModel                9

In [3]:
# عرض الأعمدة المفيدة فقط (اللي نسبة الـ Null فيها أقل من 60%) بالكامل بدون اختصار
useful_cols = summary[summary['Null_%'] < 60]
print(f"عدد الأعمدة المفيدة: {len(useful_cols)} من أصل 76\n")

for idx, row in useful_cols.iterrows():
    print(f"{idx:<30} | Null: {row['Null_%']:>5}% | Sample: {row['Sample']}")

عدد الأعمدة المفيدة: 25 من أصل 76

OperatorControlNumber          | Null:   0.0% | Sample: P5CA2019010300001
DifficultyDate                 | Null:   0.0% | Sample: 01/03/2019
SubmissionDate                 | Null:   0.0% | Sample: 2019-01-03T11:59:51.033-05:00
OperatorDesignator             | Null:   2.2% | Sample: P5CA
SubmitterDesignator            | Null:   1.8% | Sample: P5CA
SubmitterTypeCode              | Null:   0.0% | Sample: A
ReceivingRegionCode            | Null:   0.0% | Sample: EA
ReceivingDistrictOffice        | Null:   0.0% | Sample: 15
SDRType                        | Null:   0.0% | Sample: A
JASCCode                       | Null:   0.0% | Sample: 2432
NatureOfConditionA             | Null:   0.0% | Sample: J
PrecautionaryProcedureA        | Null:   0.0% | Sample: K
StageOfOperationCode           | Null:   0.0% | Sample: NR
HowDiscoveredCode              | Null:   0.0% | Sample: O
RegistryNNumber                | Null:   0.8% | Sample: 642GT
AircraftMake              